# Main Priority Model (Decision Layer)

Combines outputs from plumbing, electrical, and structural models into a final priority score.

Priority logic:
- Risk levels: Low=1, Medium=2, High=3
- Urgency = 60 - days_to_failure
- Impact weight: girls_toilet=3, classroom=2, storage=1
- Priority = sum(risk x urgency x weight), normalized to 0-100

Run all previous notebooks first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, joblib, json, warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Load All Models

In [ ]:
plumbing_model = joblib.load('../models/plumbing_model.pkl')
plumbing_features = joblib.load('../models/plumbing_features.pkl')
print(f'Plumbing model loaded ({len(plumbing_features)} features)')

electrical_model = joblib.load('../models/electrical_model.pkl')
electrical_features = joblib.load('../models/electrical_features.pkl')
print(f'Electrical model loaded ({len(electrical_features)} features)')

structural_model = joblib.load('../models/structural_model.pkl')
structural_features = joblib.load('../models/structural_features.pkl')
structural_le = joblib.load('../models/structural_label_encoder.pkl')
print(f'Structural model loaded ({len(structural_features)} features)')

df_full = pd.read_csv('../data/preprocessed/full_preprocessed.csv')
print(f'\nFull dataset: {df_full.shape}')

In [ ]:
# show saved metrics
for name in ['plumbing', 'electrical', 'structural']:
    with open(f'../models/{name}_metrics.json') as f:
        m = json.load(f)
    print(f'\n{name.upper()}:')
    print(f'  Accuracy: {m.get("accuracy", "N/A")}')
    if 'f1_score' in m:
        print(f'  F1:       {m["f1_score"]}')
    if 'weighted_f1' in m:
        print(f'  F1 (wt):  {m["weighted_f1"]}')

## 2. Priority Logic Functions

In [ ]:
def prob_to_risk(prob):
    if prob >= 0.7: return 'High'
    if prob >= 0.4: return 'Medium'
    return 'Low'

def risk_value(risk):
    return {'Low': 1, 'Medium': 2, 'High': 3}.get(risk, 1)

def severity_to_risk(sev):
    return {'Safe': 'Low', 'Warning': 'Medium', 'Danger': 'High'}.get(sev, 'Low')

def impact_weight(category, girls_school):
    if category == 'plumbing' and girls_school:
        return 3
    if category in ['plumbing', 'electrical']:
        return 2
    return 1

def priority_score(risk_val, urgency, weight):
    raw = risk_val * max(0, urgency) * weight
    return min(100, round(raw / (3 * 60 * 3) * 100, 2))

def priority_label(score):
    if score >= 75: return 'Critical'
    if score >= 50: return 'High'
    if score >= 25: return 'Medium'
    return 'Low'

print('Priority functions defined')

## 3. predict_all() Function

In [ ]:
def predict_all(input_data, verbose=True):
    """
    Runs all 3 models and computes final priority.
    
    Returns dict with:
    - risks per category
    - days to failure
    - final priority score
    - explanation
    """
    if isinstance(input_data, dict):
        input_data = pd.Series(input_data)
    
    category = input_data.get('category', 'unknown')
    girls = int(input_data.get('girls_school', 0))
    students = float(input_data.get('num_students', 0))
    dtf = float(input_data.get('days_to_failure', 60))
    
    results = {'school_id': input_data.get('school_id', 'N/A'), 'category': category}
    
    # plumbing prediction
    try:
        p_in = input_data[plumbing_features].fillna(0).values.reshape(1, -1)
        p_prob = plumbing_model.predict_proba(p_in)[0][1]
        p_risk = prob_to_risk(p_prob)
    except:
        p_prob, p_risk = 0.0, 'Low'
    
    # electrical prediction
    try:
        e_in = input_data[electrical_features].fillna(0).values.reshape(1, -1)
        e_prob = electrical_model.predict_proba(e_in)[0][1]
        e_risk = prob_to_risk(e_prob)
    except:
        e_prob, e_risk = 0.0, 'Low'
    
    # structural prediction
    try:
        s_in = input_data[structural_features].fillna(0).values.reshape(1, -1)
        s_pred = structural_model.predict(s_in)[0]
        s_label = structural_le.inverse_transform([s_pred])[0]
        s_risk = severity_to_risk(s_label)
    except:
        s_label, s_risk = 'Safe', 'Low'
    
    results['category_risks'] = {
        'plumbing': {'probability': round(float(p_prob), 4), 'risk': p_risk},
        'electrical': {'probability': round(float(e_prob), 4), 'risk': e_risk},
        'structural': {'severity': s_label, 'risk': s_risk},
    }
    
    # compute final priority
    urgency = max(0, 60 - dtf)
    
    cat_scores = []
    for cat, risk in [('plumbing', p_risk), ('electrical', e_risk), ('structural', s_risk)]:
        rv = risk_value(risk)
        wt = impact_weight(cat, girls)
        sc = priority_score(rv, urgency, wt)
        cat_scores.append({'category': cat, 'risk_value': rv, 'weight': wt, 'score': sc})
    
    total = sum(c['score'] for c in cat_scores)
    
    # weight the actual category more
    if category in ['plumbing', 'electrical', 'structural']:
        idx = {'plumbing': 0, 'electrical': 1, 'structural': 2}[category]
        final = min(100, round(cat_scores[idx]['score'] * 0.6 + total * 0.4 / 3, 2))
    else:
        final = min(100, round(total / 3 * 1.5, 2))
    
    label = priority_label(final)
    
    results['priority'] = {
        'score': final,
        'label': label,
        'days_to_failure': round(dtf, 2),
        'urgency': round(urgency, 2),
        'breakdown': cat_scores,
    }
    
    # build explanation
    parts = []
    risk_map = {'plumbing': p_risk, 'electrical': e_risk, 'structural': s_risk}
    high = [c for c, r in risk_map.items() if r == 'High']
    med = [c for c, r in risk_map.items() if r == 'Medium']
    if high: parts.append(f"HIGH risk in {', '.join(high)}")
    if med: parts.append(f"MEDIUM risk in {', '.join(med)}")
    if girls: parts.append('girls school (priority boost)')
    if urgency > 40: parts.append(f'urgent ({dtf:.0f} days to failure)')
    if students > 1000: parts.append(f'high impact ({students:.0f} students)')
    
    results['explanation'] = f"{label} priority (score: {final}/100) - {'; '.join(parts)}."
    
    if verbose:
        print(f'\nSchool {results["school_id"]} ({category})')
        print(f'  Plumbing:    {p_risk} (prob: {p_prob:.3f})')
        print(f'  Electrical:  {e_risk} (prob: {e_prob:.3f})')
        print(f'  Structural:  {s_risk} ({s_label})')
        print(f'  Priority:    {final}/100 ({label})')
        print(f'  {results["explanation"]}')
    
    return results

print('predict_all() defined')

## 4. Test Predictions

In [ ]:
# test with one sample from each category
for cat in ['plumbing', 'electrical', 'structural']:
    sample = df_full[df_full['category'] == cat].iloc[0]
    predict_all(sample)
    print('---')

## 5. Batch Predictions

In [ ]:
sample_df = df_full.sample(n=500, random_state=42)

batch = []
for _, row in sample_df.iterrows():
    r = predict_all(row, verbose=False)
    batch.append({
        'school_id': r['school_id'],
        'category': r['category'],
        'plumbing_risk': r['category_risks']['plumbing']['risk'],
        'electrical_risk': r['category_risks']['electrical']['risk'],
        'structural_severity': r['category_risks']['structural']['severity'],
        'priority_score': r['priority']['score'],
        'priority_label': r['priority']['label'],
    })

results_df = pd.DataFrame(batch)
print(f'Batch done: {len(results_df)} records')
print(f'\nPriority distribution:')
print(results_df['priority_label'].value_counts())
results_df.head(10)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

order = ['Low', 'Medium', 'High', 'Critical']
cols = ['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']
lc = results_df['priority_label'].value_counts().reindex(order, fill_value=0)
axes[0,0].bar(order, lc.values, color=cols)
axes[0,0].set_title('Priority Label Distribution')

sns.histplot(results_df['priority_score'], bins=30, ax=axes[0,1], color='#9b59b6')
axes[0,1].set_title('Priority Score Distribution')

ro = ['Low', 'Medium', 'High']
pc = results_df['plumbing_risk'].value_counts().reindex(ro, fill_value=0)
axes[0,2].bar(ro, pc.values, color=['#2ecc71','#f39c12','#e74c3c'])
axes[0,2].set_title('Plumbing Risk')

ec = results_df['electrical_risk'].value_counts().reindex(ro, fill_value=0)
axes[1,0].bar(ro, ec.values, color=['#2ecc71','#f39c12','#e74c3c'])
axes[1,0].set_title('Electrical Risk')

so = ['Safe', 'Warning', 'Danger']
sc = results_df['structural_severity'].value_counts().reindex(so, fill_value=0)
axes[1,1].bar(so, sc.values, color=['#2ecc71','#f39c12','#e74c3c'])
axes[1,1].set_title('Structural Severity')

sns.boxplot(data=results_df, x='category', y='priority_score', ax=axes[1,2], palette='Set2')
axes[1,2].set_title('Priority by Category')

plt.tight_layout()
plt.show()

## 6. Example Input/Output JSON

In [ ]:
# pick a high-priority girls school example
sample = df_full[(df_full['failure_within_30_days']==1) & (df_full['girls_school']==1)].iloc[0]
result = predict_all(sample, verbose=False)

print('Example Input (key fields):')
inp = {
    'school_id': int(sample.get('school_id', 0)),
    'category': str(sample.get('category', '')),
    'girls_school': int(sample.get('girls_school', 0)),
    'num_students': int(sample.get('num_students', 0)),
    'building_age': float(sample.get('building_age', 0)),
    'condition_score': float(sample.get('condition_score', 0)),
    'water_leak': int(sample.get('water_leak', 0)),
    'wiring_exposed': int(sample.get('wiring_exposed', 0)),
    'crack_width_mm': float(sample.get('crack_width_mm', 0)),
    'days_to_failure': float(sample.get('days_to_failure', 0)),
}
print(json.dumps(inp, indent=2))

print('\nExample Output:')
print(json.dumps(result, indent=2, default=str))

## 7. API-Ready Function

In [ ]:
def predict_api(input_json):
    """API wrapper. Pass a dict, get back structured response."""
    try:
        result = predict_all(pd.Series(input_json), verbose=False)
        return {
            'success': True,
            'data': {
                'school_id': result['school_id'],
                'risks': {
                    'plumbing': result['category_risks']['plumbing']['risk'],
                    'electrical': result['category_risks']['electrical']['risk'],
                    'structural': result['category_risks']['structural']['severity'],
                },
                'priority': result['priority'],
                'explanation': result['explanation'],
            }
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

api_result = predict_api(sample.to_dict())
print('API response:')
print(json.dumps(api_result, indent=2, default=str))

## 8. Save Pipeline Config

In [ ]:
config = {
    'pipeline': 'InfraRakshak Predictive Maintenance',
    'version': '1.0.0',
    'models': {
        'plumbing': {'path': 'models/plumbing_model.pkl', 'type': 'RandomForest', 'target': 'failure_within_30_days'},
        'electrical': {'path': 'models/electrical_model.pkl', 'type': 'RandomForest', 'target': 'failure_within_30_days'},
        'structural': {'path': 'models/structural_model.pkl', 'type': 'RandomForest', 'target': 'structural_severity'},
    },
    'priority_logic': {
        'risk_weights': {'Low': 1, 'Medium': 2, 'High': 3},
        'impact_weights': {'girls_toilet': 3, 'classroom': 2, 'storage': 1},
        'urgency_formula': '60 - days_to_failure',
        'thresholds': {'Critical': 75, 'High': 50, 'Medium': 25, 'Low': 0},
    }
}

with open('../models/pipeline_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Pipeline config saved')
print('\nAll files:')
for f in os.listdir('../models'):
    print(f'  {f}')